# Autonomous 360 Video Generator

**Author:** Peter Rutledge  
**Original course submission:** May 9, 2026

Concepts demonstrated in this project:
Increasing familiarity with chosen framework/harness
I implemented the Orchestration as sequential Python rather than LangGraph — a deliberate choice given the pipeline's strict linear dependencies, with stateless dict-passing nodes that keep a graph-framework migration path open. Different models are assigned to different roles (llama3.2 for planning and social writing, qwen2.5-coder:7b for Blender code generation) so each agent's strengths match its task.

Experiments with prompting:
For the scene generation, I am using a single agent that reads from the ChromaDB and uses a novelty gate to determine if the scene the propose is too similar to what is in the database.  There are three re-rolls to generate new scene ideas.
I am using an Agent to Agent (A2A) social media text generation debate with three personas at different temperatures.  A Generator, a Critic, and a Decider.  The Generator reads in from the vector database what copy text we already created and comes up with a new original copy text draft.  Then the Critic makes suggestions to improve the phrasing.  Last the Decider makes the final decision on what the text will be.

Experiments with multi-agent orchestration:
I created a mixed model swarm.  llama3.2 for planning and post copy.  qwen2.5-coder:7b to write the blender code.  And a separate agent to generate the music (deterministic).

Experiments with memory:
I used a ChromaDB for both scene planning using memory and a novelty gate.  And for the social media posts.  This ensures we don't get duplication in new scenes.  The novelty gate is interesting as I do a re-roll upto three times if the ideas are too similar to what is in the vector database.

Experiments with tool use:
Blender MCP was the main tool I used.  I also used ffmpeg, Puppeteer, and npm scripts.  I also used a proven safe template (environment or entity) as the primary script. Ollama-generated Blender Python is too unreliable (hallucinated API calls, Blender 3.x/4.x naming differences) to use directly. (asset_pipeline.py)

Experiments with safeguards:
Human in the loop publishing.  The swarm generates everything, but I make the final call for what gets posted to YouTube and the social media platforms, and post manually.
Novelty Gates.  Prevents the swarm from publishing near duplicate content.
Safe Code Templates.  As described above I had templates to ensure Ollama could produce reliable code.
Smoke tests.  The pipeline will fail if Blender MCP is not available.
Validation steps to ensure the file size is greater than 0.
Cleanup at start of next run ensures that I keep artifacts at the end of a run.

Alignment with course objectives:
Memory was a ChromaDB with two separate collections
Tool Use - Blender MCP, Three.js and ffmpeg
Multi agent orchestration was done by sequential python due to the sequential nature of this workflow.  I considered LangGraph but decided that it would add dependency weight without enabling any capability improvement.
Prompting was A2A debate.
Safeguards was by human in the loop, novelty gates, safe templates, and a smoke test.


Introducing Autonomous 360 Video Generator.  An autonomous multi-agent pipeline that generates daily 2-minute 360° equirectangular VR videos with a retro
low-poly aesthetic and prepares them for publishing to a dedicated YouTube channel. The system runs autonomously from
scene imagination through 3D asset creation, physics simulation, procedural music generation, and social post design. Final
publishing is a deliberate human step as an intentional guardrail — all generated content is manually reviewed before content reaches a public audience.

- **llama3.2** → scene plan + post copy generation (creator persona)
- **qwen2.5-coder:7b** → Blender Python for entity assets
- **Blender MCP socket** → executes the code, exports GLBs

### Flow
1. Smoke-test the Blender MCP socket
2. Query ChromaDB scene memory
3. llama3.2 proposes a scene (novelty gate)
4. qwen2.5-coder writes Blender Python for each entity
5. Blender exports GLBs via MCP socket
6. Music + manifest assembly + ChromaDB store
7. Render the 360° MP4
8. The studio assistant writes post copy for X, Facebook, Instagram, YouTube** (saved as JSON)

## Imports

In [1]:
from __future__ import annotations

import importlib
import json
import os
import random as _rng
import re
import subprocess
import sys
import urllib.request
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.asset_pipeline as _ap_mod
import scripts.manifest_builder as _mb_mod
importlib.reload(_ap_mod)
importlib.reload(_mb_mod)

from scripts.asset_pipeline import (
    absolute_asset_path,
    build_asset_tasks,
    default_service_host,
    execute_blender_mcp_code,
    hex_to_rgba,
    validate_asset_tasks,
    windows_unc_path,
)
from scripts.manifest_builder import build_manifest, save_manifest
from scripts.scene_memory import SceneMemory

print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"asset_pipeline : {_ap_mod.__file__}")

PROJECT_ROOT   : <project-root>
asset_pipeline : <project-root>/scripts/asset_pipeline.py


## Configuration

In [2]:
# --- Scene ---
CANDIDATE_SCENE       = ""     # leave empty for fully autonomous
RENDER_DURATION_SECS  = 120
RENDER_FPS            = 30
RENDER_WIDTH          = 3840
RENDER_HEIGHT         = 1920
RUN_RENDER            = True

# --- Novelty gate ---
SCENE_MEMORY_TOP_K    = 5
SCENE_MEMORY_THRESHOLD = 0.35
NOVELTY_MAX_ATTEMPTS  = 3      # re-roll up to this many times if scene is too similar
STORE_IN_MEMORY       = True

# --- Local services ---
# Under WSL, automatically resolves the Windows host from the default route.
# Set OLLAMA_BASE_URL or A360VG_MCP_HOST to override auto-detection.
SERVICE_HOST = default_service_host()

# --- Models ---
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", f"http://{SERVICE_HOST}:11434")
PLANNER_MODEL   = "llama3.2:latest"    # scene concept + mood + colors (JSON)
CODER_MODEL     = "qwen2.5-coder:7b"   # Blender Python (code)
OLLAMA_TIMEOUT  = 120
CODER_TIMEOUT   = 180              # qwen may need more time for complex scripts

# --- Blender MCP ---
MCP_HOST          = os.environ.get("A360VG_MCP_HOST", SERVICE_HOST)
MCP_PORT          = os.environ.get("A360VG_MCP_PORT", "9876")
MCP_ASSET_TIMEOUT = 120

# --- Daily variation ---
_THEMES = [
    "arctic tundra", "deep ocean trench", "ancient forest", "volcanic island",
    "floating sky islands", "desert canyon", "foggy marshland", "crystal cavern",
    "bamboo grove", "frozen lake", "red planet surface", "cloud kingdom",
    "sunken ruins", "neon city debris", "bioluminescent reef", "mountain glacier",
    "autumn meadow", "lava fields", "dense jungle", "midnight swamp",
    "aurora plains", "salt flats", "undersea temple", "wind-carved cliffs",
]
DAILY_THEME = _rng.choice(_THEMES)
MUSIC_SEED  = _rng.randint(0, 99999)

os.environ["A360VG_ASSET_EXECUTOR"]     = "blender_mcp_direct"
os.environ["A360VG_MCP_HOST"]           = MCP_HOST
os.environ["A360VG_MCP_PORT"]           = MCP_PORT
os.environ["A360VG_MCP_MESSAGE_FORMAT"] = "execute_code"
os.environ["OLLAMA_BASE_URL"]            = OLLAMA_BASE_URL
os.environ["OLLAMA_MODEL"]               = PLANNER_MODEL
os.environ["OLLAMA_TIMEOUT"]             = str(OLLAMA_TIMEOUT)
os.environ["A360VG_MAX_AUTORUNS"]       = "10"

SCENE_ID = f"scene-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
print(f"Scene ID    : {SCENE_ID}")
print(f"Daily theme : {DAILY_THEME}")
print(f"Music seed  : {MUSIC_SEED}")
print(f"Planner     : {PLANNER_MODEL}")
print(f"Coder       : {CODER_MODEL}")
print(f"MCP         : {MCP_HOST}:{MCP_PORT}")

Scene ID    : scene-20261025-123453
Daily theme : bamboo grove
Music seed  : 58765
Planner     : llama3.2:latest
Coder       : qwen2.5-coder:7b
MCP         : 127.0.0.1:9876


## Smoke Test

In [3]:
smoke = execute_blender_mcp_code(
    "print('10 pipeline smoke test ok')",
    expect_response=True,
    timeout_seconds=15,
)
print(smoke)
assert smoke.get("status") == "success", (
    "Blender MCP socket not reachable — is Blender open with the MCP addon active?"
)

{'status': 'success', 'host': '127.0.0.1', 'port': 9876, 'protocol': 'proven_raw_json', 'response': {'status': 'success', 'result': {'executed': True, 'result': '10 pipeline smoke test ok\n'}}, 'raw_response': ''}


## Cleanup — Wipe Artifacts From Previous Runs

Deletes the intermediate files from the last run so the workspace stays fresh.
Runs AFTER the smoke test so a broken MCP setup doesn't wipe your inspectable artifacts.

**Preserved:** `.video-generator-memory/` (ChromaDB scene + post memory — the brain)

**Wiped:** manifests, generated GLBs, audio, frames, qwen scripts, post JSON, old MP4s

In [4]:
# --- Clean up artifacts from previous runs ---
import shutil


def _remove_files(root, patterns):
    """Remove files matching glob patterns under `root`. Returns count."""
    if not root.exists():
        return 0
    count = 0
    for pattern in patterns:
        for p in root.glob(pattern):
            if p.is_file():
                p.unlink()
                count += 1
    return count


def _empty_dir(d):
    """Delete all contents of a directory but keep the directory itself. Returns count."""
    if not d.exists():
        return 0
    count = 0
    for p in d.iterdir():
        if p.name == ".gitkeep":
            continue
        if p.is_file():
            p.unlink()
            count += 1
        elif p.is_dir():
            shutil.rmtree(p)
            count += 1
    return count


totals = {}

# Scene-specific files in public/ — match the patterns from .gitignore
totals["manifests"]    = _remove_files(PROJECT_ROOT / "public/manifests",
                                        ["scene-*.json"])
totals["environments"] = _remove_files(PROJECT_ROOT / "public/assets/environments",
                                        ["scene-*", "*_environment.glb", "*_environment.glb.log"])
totals["audio"]        = _remove_files(PROJECT_ROOT / "public/assets/audio",
                                        ["scene-*.wav", "scene-*.mid"])

# Fully gitignored directories — safe to empty completely
totals["generated"]      = _empty_dir(PROJECT_ROOT / "public/assets/generated")
totals["posts"]          = _empty_dir(PROJECT_ROOT / "public/posts")

# Output directories
totals["frames"]         = _empty_dir(PROJECT_ROOT / "output/frames")
totals["qwen_scripts"]   = _empty_dir(PROJECT_ROOT / "output/qwen-blender-scripts")
totals["ollama_scripts"] = _empty_dir(PROJECT_ROOT / "output/ollama-blender-scripts")
totals["rendered_mp4s"]  = _remove_files(PROJECT_ROOT / "output", ["*.mp4"])

print("Cleaned up artifacts from previous runs:")
for name, count in totals.items():
    mark = "." if count == 0 else "-"
    print(f"  [{mark}] {name:16s} : {count:>4}")

print()
print("Preserved : .video-generator-memory/ (ChromaDB scene + post memory)")

Cleaned up artifacts from previous runs:
  [.] manifests        :    0
  [.] environments     :    0
  [.] audio            :    0
  [.] generated        :    0
  [.] posts            :    0
  [.] frames           :    0
  [-] qwen_scripts     :    1
  [.] ollama_scripts   :    0
  [.] rendered_mp4s    :    0

Preserved : .video-generator-memory/ (ChromaDB scene + post memory)


## Scene Memory Query

In [5]:
memory = SceneMemory()
prior_count = memory.collection.count()

prior_scenes_for_prompt = []

if prior_count > 0:
    raw = memory.collection.get(include=["metadatas", "documents"])
    ids   = raw.get("ids", [])
    metas = raw.get("metadatas") or []
    docs  = raw.get("documents") or []
    scenes = []
    for i, sid in enumerate(ids):
        scenes.append({
            "scene_id": sid,
            "metadata": metas[i] if i < len(metas) else {},
            "document": docs[i]  if i < len(docs)  else "",
        })
    scenes.sort(key=lambda s: s["metadata"].get("stored_at", ""), reverse=True)
    prior_scenes_for_prompt = scenes[:SCENE_MEMORY_TOP_K]

print(f"Scenes in memory : {prior_count}")
for m in prior_scenes_for_prompt:
    meta = m.get("metadata", {})
    print(f"  [{meta.get('stored_at','?')[:10]}] {m['scene_id']} — {meta.get('title','')}")

Scenes in memory : 14
  [2026-05-09] scene-20260509-063014 — Auroral Hush
  [2026-05-09] scene-20260509-055747 — Moonlit Mire
  [2026-05-09] scene-20260509-054316 — Shadows in the Emerald Canopy
  [2026-04-28] scene-20260428-200546 — Salting the Wastes of Eternity
  [2026-04-22] scene-20260422-212350 — Misty Dawn on the Marsh


## Scene Planning + Novelty Gate

llama3.2 proposes a scene. The novelty gate checks it against ChromaDB.
If too similar to a prior scene, a new theme is picked and Ollama tries again.
Up to `NOVELTY_MAX_ATTEMPTS` re-rolls before accepting whatever it has.

In [6]:
PLANNER_SYSTEM = """You are a scene planner for Autonomous 360 Video Generator, an autonomous low-poly VR video channel.
Return ONLY valid JSON with no markdown fences and no extra text.

Output exactly this schema:
{
  "scene_id": "<lowercase-hyphenated-id>",
  "title": "<scene title>",
  "scene_description": "<one sentence>",
  "mood": "<one of: dreamy, mysterious, dark, cold, peaceful>",
  "environment_asset": {
    "action": "generate_new",
    "description": "<describe the sky and ground in one sentence>",
    "sky_color": "<hex e.g. #1a3a5c>",
    "ground_color": "<hex e.g. #2d4a1e>"
  },
  "entity_assets": [
    {
      "asset_id": "<lowercase-hyphenated-id>",
      "action": "generate_new",
      "description": "<concrete visual description — what does it look like, what is it made of>",
      "color": "<primary hex color>",
      "shape": "<one of: ico_sphere, cube, cone, cylinder, torus>"
    }
  ]
}

Rules:
- 3 to 5 entity_assets — they will drift through the scene one at a time, with new ones still entering during the second half of the video
- Low-poly PS1 aesthetic, solid flat colors
- entity description must be specific enough to build in 3D (e.g. 'a round hot air balloon with a small square basket hanging below it')
- All hex colors must be valid 6-digit codes
- The scene must be clearly different from any prior scenes listed"""


def _call_planner(theme, prior_lines):
    prior_block = (
        "Prior scenes (create something clearly different):\n" + "\n".join(prior_lines)
        if prior_lines else "No prior scenes."
    )
    user_msg = (
        f"{prior_block}\n\n"
        + (f"Candidate theme: {CANDIDATE_SCENE}" if CANDIDATE_SCENE
           else f"Today's setting: {theme}. Build a scene inspired by this.")
    )
    body = json.dumps({
        "model": PLANNER_MODEL,
        "messages": [
            {"role": "system", "content": PLANNER_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        "stream": False,
        "format": "json",
        "options": {"temperature": 0.9},
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_BASE_URL}/api/chat", data=body,
        headers={"Content-Type": "application/json"}, method="POST",
    )
    with urllib.request.urlopen(req, timeout=OLLAMA_TIMEOUT) as resp:
        raw = json.loads(resp.read())["message"]["content"].strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```[a-zA-Z]*\n?", "", raw).rstrip()
        if raw.endswith("```"): raw = raw[:-3].rstrip()
    try:
        plan = json.loads(raw)
    except json.JSONDecodeError:
        print("PLANNER raw output:\n", raw)
        raise
    plan["scene_id"] = SCENE_ID
    return plan


# Build prior-scene lines once
prior_lines = []
for m in prior_scenes_for_prompt:
    meta  = m.get("metadata", {})
    title = meta.get("title") or m["scene_id"]
    doc   = m.get("document", "")
    desc_match = re.search(r"description: (.+)", doc)
    desc = desc_match.group(1).strip() if desc_match else ""
    prior_lines.append(f'- "{title}"' + (f": {desc}" if desc else ""))

# --- Novelty-gated planning loop ---
scene_plan   = None
current_theme = DAILY_THEME

for attempt in range(1, NOVELTY_MAX_ATTEMPTS + 1):
    print(f"\nAttempt {attempt}/{NOVELTY_MAX_ATTEMPTS}  theme: {current_theme}")
    scene_plan = _call_planner(current_theme, prior_lines)
    title = scene_plan.get("title", "?")
    desc  = scene_plan.get("scene_description", "")
    print(f"  Ollama proposed : {title}")

    if prior_count == 0:
        print("  Novelty check   : memory empty — accepted")
        break

    gate = memory.novelty_gate(
        desc, top_k=SCENE_MEMORY_TOP_K, similarity_threshold=SCENE_MEMORY_THRESHOLD
    )
    sim = gate["max_similarity"]

    if gate["novel"]:
        print(f"  Novelty check   : similarity={sim:.2f} — NOVEL, accepted")
        break
    else:
        closest = gate["matches"][0]["metadata"].get("title", "?") if gate["matches"] else "?"
        print(f"  Novelty check   : similarity={sim:.2f} — too similar to '{closest}', re-rolling")
        current_theme = _rng.choice([t for t in _THEMES if t != current_theme])
else:
    print("  Max attempts reached — proceeding with last proposal")

env = scene_plan.get("environment_asset", {})
print(f"\nApproved scene  : {scene_plan.get('title')}  [{scene_plan.get('mood')}]")
print(f"Sky / ground    : {env.get('sky_color')} / {env.get('ground_color')}")
for e in scene_plan.get("entity_assets", []):
    print(f"  {e['asset_id']:30s}  {e.get('color')}  {e.get('shape')}  — {e.get('description','')}")


Attempt 1/3  theme: bamboo grove
  Ollama proposed : Bamboo Sanctuary
  Novelty check   : similarity=0.19 — NOVEL, accepted

Approved scene  : Bamboo Sanctuary  [peaceful]
Sky / ground    : #87CEEB / #6495ED
  bamboo-stalk                    #C6E2B5  cylinder  — A slender bamboo stalk with smooth, pale green skin, swaying gently in the breeze
  leaf-gatherer                   #34C759  torus  — A delicate, hand-woven basket made from intertwined bamboo leaves, suspended from a slender branch
  bamboo-blooms                   #FFC0CB  ico_sphere  — A cluster of vibrant, pink-tinged bamboo blooms, swaying softly in the sunlight
  sun-disk                        #F7DC6F  circle  — A bright, circular sun disk suspended in the sky, casting dappled shadows on the grove floor
  dappled-shade                   #B2E6CE  torus  — A soft, gradient-filled area of dappled shade, providing a cool respite from the sun's warm rays


## Asset Tasks

In [7]:
asset_tasks = build_asset_tasks(scene_plan)

print(f"{len(asset_tasks)} task(s):")
for t in asset_tasks:
    print(f"  [{t['task_type']:12s}] {t['asset_id']:35s} {t['action']}")
    print(f"    → {t['target_path']}")

6 task(s):
  [environment ] scene-20261025-123453_environment   generate_new
    → public/assets/environments/scene-20261025-123453_environment.glb
  [entity      ] bamboo-stalk                        generate_new
    → public/assets/generated/bamboo-stalk.glb
  [entity      ] leaf-gatherer                       generate_new
    → public/assets/generated/leaf-gatherer.glb
  [entity      ] bamboo-blooms                       generate_new
    → public/assets/generated/bamboo-blooms.glb
  [entity      ] sun-disk                            generate_new
    → public/assets/generated/sun-disk.glb
  [entity      ] dappled-shade                       generate_new
    → public/assets/generated/dappled-shade.glb


## Code Generation — qwen2.5-coder

For each asset, qwen2.5-coder writes a complete Blender Python script from scratch
given the asset description and a few-shot example.

Scripts are saved to `output/qwen-blender-scripts/<scene_id>/` for inspection.

In [8]:
import scripts.asset_pipeline as ap
importlib.reload(ap)

# --- Few-shot example shown to qwen for every entity asset ---
# Hot air balloon: large sphere (envelope) + small cube (basket), two materials, joined.
# Demonstrates: multi-primitive composition, material helper, join, GLB export.
EXAMPLE_CODE = """\
import bpy, os

bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete(use_global=False)

# balloon envelope
bpy.ops.mesh.primitive_ico_sphere_add(subdivisions=3, radius=1.0, location=(0, 0, 1.2))
balloon = bpy.context.active_object
balloon.name = 'balloon'
balloon.scale = (1.0, 1.0, 1.25)

# basket
bpy.ops.mesh.primitive_cube_add(size=1.0, location=(0, 0, -0.2))
basket = bpy.context.active_object
basket.name = 'basket'
basket.scale = (0.35, 0.35, 0.22)

def _mat(name, rgba):
    m = bpy.data.materials.new(name)
    m.use_nodes = True
    b = m.node_tree.nodes['Principled BSDF']
    b.inputs['Base Color'].default_value = rgba
    b.inputs['Roughness'].default_value = 1.0
    for k in ('Specular', 'Specular IOR Level'):
        if k in b.inputs:
            b.inputs[k].default_value = 0.0
            break
    return m

balloon.data.materials.append(_mat('balloon_mat', (0.9, 0.2, 0.1, 1.0)))
basket.data.materials.append(_mat('basket_mat', (0.5, 0.35, 0.15, 1.0)))

bpy.ops.object.select_all(action='DESELECT')
balloon.select_set(True)
basket.select_set(True)
bpy.context.view_layer.objects.active = balloon
bpy.ops.object.join()

_p = r"\\\\wsl$\\Ubuntu\\path\\to\\balloon.glb"
os.makedirs(os.path.dirname(_p), exist_ok=True)
bpy.ops.export_scene.gltf(filepath=_p, export_format='GLB', export_apply=True,
    export_materials='EXPORT', export_cameras=False, export_lights=False)
"""

CODER_SYSTEM = """You are a Blender Python expert writing low-poly 3D game assets.

RULES — follow exactly:
- Import only bpy and os
- First two lines: clear the scene (select_all + delete)
- Build the asset from 2-4 primitives composed to make it recognisable
  (balloon + basket, tree trunk + cone crown, lighthouse base + cylinder tower, etc.)
- Primitives: ico_sphere, uv_sphere, cube, cone, cylinder, torus only
- Material helper must try both 'Specular' AND 'Specular IOR Level' (Blender 3.x/4.x compat)
- Assign a different material to each part before joining
- Join all parts with bpy.ops.object.join() before export
- Export path comes from the task — copy it EXACTLY into the script as a raw string r\"...\"
- Return ONLY Python code — no markdown, no explanation

FORBIDDEN — these break Blender:
- Do NOT use bpy.ops.object.duplicate() or duplicate loops. Build ONE single asset, not multiple copies.
- Do NOT use `obj.location += (x, y, z)` — tuples cannot be added to Vectors. Use `obj.location.x += value` instead.
- Do NOT import mathutils, Vector, or any module other than bpy and os.
- Do NOT use for loops to generate geometry — write each primitive call out explicitly.
- Do NOT create images, textures, or shader nodes (no bpy.ops.image.new, ShaderNodeTexImage, ShaderNodeMath, etc.).
- Do NOT access bsdf.inputs[\"Specular\"] directly. ONLY use the `_mat(name, rgba)` helper from the example for materials. The helper safely handles Blender 3.x vs 4.x naming.
- Do NOT load or reference any external files (no .png, .jpg, .obj imports). Solid colors only.
- Materials must be SOLID FLAT COLORS via the _mat helper. Nothing else."""



# --- Pre-flight pattern check: catch obviously broken qwen output before sending to Blender ---
FORBIDDEN_PATTERNS = [
    (r"\bfor\s+\w+\s+in\s+range",   "uses a for loop with range()"),
    (r"\brandom\.",                    "uses the random module (not imported, not allowed)"),
    (r"\bmath\.|\bsin\(|\bcos\(|\bpi\b",  "uses math module / pi / sin / cos"),
    (r"^\s*import\s+(?!bpy|os\b)\w+",      "imports a module other than bpy and os"),
    (r"^\s*from\s+\w+\s+import",     "uses a from-import"),
    (r"bpy\.ops\.image\.new",         "creates an image (forbidden)"),
    (r"ShaderNodeTexImage",              "creates a texture node (forbidden)"),
    (r"\.location\s*\+=\s*\(",       "uses obj.location += (tuple) (invalid Python)"),
    (r"bpy\.ops\.object\.duplicate",  "uses bpy.ops.object.duplicate"),
    (r"bsdf\.inputs\[['\"]Specular['\"]\]\.default_value", "sets bsdf.inputs['Specular'] directly without the for-k fallback"),
]


def _check_forbidden(code):
    """Return a list of forbidden-pattern issue strings found in the qwen code."""
    issues = []
    for pattern, msg in FORBIDDEN_PATTERNS:
        if re.search(pattern, code, re.MULTILINE):
            issues.append(msg)
    return issues


def _qwen_generate(task):
    """Ask qwen2.5-coder to write Blender Python for one entity asset task."""
    linux_path = absolute_asset_path(task["asset_path"])
    unc        = windows_unc_path(linux_path)
    desc       = task.get("description", "a simple low-poly object")
    color_hex  = task.get("color") or "#6699cc"
    rgba       = hex_to_rgba(color_hex)
    color_str  = f"({rgba[0]}, {rgba[1]}, {rgba[2]}, 1.0)"

    user_msg = (
        f"Here is a working example (hot air balloon with basket):\n\n"
        f"{EXAMPLE_CODE}\n"
        f"---\n"
        f"Now write a NEW script for this asset:\n\n"
        f"Asset: {desc}\n"
        f"Primary color: {color_str}  (hex: {color_hex})\n"
        f"Scale: roughly 1-2 units tall, centred at origin.\n\n"
        f"Export path (copy EXACTLY into the script):\n{unc}\n\n"
        f"Write the complete bpy script now:"
    )

    body = json.dumps({
        "model": CODER_MODEL,
        "messages": [
            {"role": "system", "content": CODER_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        "stream": False,
        "options": {"temperature": 0.2},
    }).encode()

    req = urllib.request.Request(
        f"{OLLAMA_BASE_URL}/api/chat", data=body,
        headers={"Content-Type": "application/json"}, method="POST",
    )
    with urllib.request.urlopen(req, timeout=CODER_TIMEOUT) as resp:
        code = json.loads(resp.read())["message"]["content"].strip()

    if code.startswith("```"):
        code = re.sub(r"^```[a-zA-Z]*\n?", "", code).rstrip()
        if code.endswith("```"):
            code = code[:-3].rstrip()

    return code, linux_path, unc


# --- Generate each asset ---
script_dir = PROJECT_ROOT / "output" / "qwen-blender-scripts" / SCENE_ID
script_dir.mkdir(parents=True, exist_ok=True)

gen_results = []

for task in asset_tasks:
    print(f"\n{'='*60}")
    print(f"Asset  : {task['asset_id']}  ({task['task_type']})")
    print(f"Desc   : {task.get('description', '')}")

    result = {
        "asset_id":  task["asset_id"],
        "task_type": task["task_type"],
        "valid":     False,
        "size":      0,
    }

    try:
        if task["task_type"] == "environment":
            # Safe template for environments — reliable sky box with correct colors
            print(f"  Using safe template  sky={task.get('sky_color')}  ground={task.get('ground_color')}")
            env_result = ap.execute_via_blender_mcp_direct(task)
            linux_path = absolute_asset_path(task["asset_path"])
            ok   = linux_path.exists() and linux_path.stat().st_size > 0
            size = linux_path.stat().st_size if ok else 0
            result.update({"valid": ok, "size": size})
            print(f"  Result : {'OK' if ok else 'FAIL'}  {size} bytes")

        else:
            # qwen2.5-coder for entity assets — retry up to 3 times if pre-flight fails
            MAX_QWEN_ATTEMPTS = 3
            code = None
            linux_path = None
            for attempt in range(1, MAX_QWEN_ATTEMPTS + 1):
                code, linux_path, unc = _qwen_generate(task)
                issues = _check_forbidden(code)
                if not issues:
                    break
                print(f"  Pre-flight attempt {attempt}: rejected — {', '.join(issues)}")
                if attempt == MAX_QWEN_ATTEMPTS:
                    print(f"  Pre-flight: giving up after {MAX_QWEN_ATTEMPTS} attempts")

            script_path = script_dir / f"{task['asset_id']}.py"
            script_path.write_text(code, encoding="utf-8")
            print(f"Script : {len(code.splitlines())} lines → {script_path.name}")
            print()
            print(code[:600] + ("\n  [...]" if len(code) > 600 else ""))
            print()

            mcp = execute_blender_mcp_code(
                code, expect_response=True, timeout_seconds=MCP_ASSET_TIMEOUT
            )

            ok   = linux_path.exists() and linux_path.stat().st_size > 0
            size = linux_path.stat().st_size if ok else 0
            result.update({"valid": ok, "size": size, "mcp_result": mcp})

            mark = "OK" if ok else "FAIL"
            print(f"Result : [{mark}]  {size} bytes  {linux_path.name}")
            if not ok:
                inner = mcp.get("response", {})
                print(f"MCP    : {str(inner)[:300]}")

    except Exception as exc:
        result["error"] = str(exc)
        print(f"ERROR  : {exc}")

    gen_results.append(result)

print(f"\n{'='*60}")
print("Summary:")
for r in gen_results:
    mark = "OK" if r["valid"] else "FAIL"
    print(f"  [{mark}] {r['asset_id']:40s} {r['size']:>8} bytes")


Asset  : scene-20261025-123453_environment  (environment)
Desc   : A tranquil bamboo grove with sunlit pathways and leafy canopies, set against a soft blue sky with a few wispy clouds
  Using safe template  sky=#87CEEB  ground=#6495ED
  Result : OK  57428 bytes

Asset  : bamboo-stalk  (entity)
Desc   : A slender bamboo stalk with smooth, pale green skin, swaying gently in the breeze
Script : 34 lines → bamboo-stalk.py

import bpy, os

bpy.ops.object.select_all(action='SELECT')
bpy.ops.object.delete(use_global=False)

# bamboo stalk
bpy.ops.mesh.primitive_cylinder_add(radius=0.1, depth=1.5, location=(0, 0, 0))
bamboo = bpy.context.active_object
bamboo.name = 'bamboo'
bamboo.scale = (0.1, 1.5, 0.1)

def _mat(name, rgba):
    m = bpy.data.materials.new(name)
    m.use_nodes = True
    b = m.node_tree.nodes['Principled BSDF']
    b.inputs['Base Color'].default_value = rgba
    b.inputs['Roughness'].default_value = 1.0
    for k in ('Specular', 'Specular IOR Level'):
        if k in b.inpu

## Validation

In [9]:
validation = validate_asset_tasks(asset_tasks)
all_valid  = all(v["valid"] for v in validation)

for v in validation:
    mark = "OK" if v["valid"] else "MISSING"
    print(f"  [{mark}] {v['asset_id']:40s} {v['size']:>8} bytes  {v['path']}")

print()
print("All assets valid:", all_valid)

  [OK] scene-20261025-123453_environment           57428 bytes  <project-root>/public/assets/environments/scene-20261025-123453_environment.glb
  [OK] bamboo-stalk                                 8224 bytes  <project-root>/public/assets/generated/bamboo-stalk.glb
  [OK] leaf-gatherer                              289112 bytes  <project-root>/public/assets/generated/leaf-gatherer.glb
  [OK] bamboo-blooms                              218816 bytes  <project-root>/public/assets/generated/bamboo-blooms.glb
  [OK] sun-disk                                    70532 bytes  <project-root>/public/assets/generated/sun-disk.glb
  [OK] dappled-shade                               70516 bytes  <project-root>/public/assets/generated/dappled-shade.glb

All assets valid: True


## Music

In [10]:
from scripts.music_agent import MusicAgent

music_result = {"wavRendered": False, "audioPath": None, "audioVolume": 0.75}
audio_dir    = PROJECT_ROOT / "public" / "assets" / "audio"

try:
    music_result = MusicAgent(output_dir=str(audio_dir)).generate(
        scene_id=scene_plan["scene_id"],
        mood=scene_plan.get("mood", "ambient"),
        duration_seconds=RENDER_DURATION_SECS,
        seed=MUSIC_SEED,
    )
    wav = audio_dir / f"{scene_plan['scene_id']}.wav"
    if music_result.get("wavRendered") and not wav.exists():
        raise RuntimeError(f"WAV missing: {wav}")
except Exception as exc:
    print(f"MUSIC ERROR: {exc}")
    music_result["wavRendered"] = False

print(f"wavRendered : {music_result.get('wavRendered')}")
print(f"audioPath   : {music_result.get('audioPath')}")

MIDI written: <project-root>/public/assets/audio/scene-20261025-123453.mid
WAV rendered: <project-root>/public/assets/audio/scene-20261025-123453.wav
wavRendered : True
audioPath   : /assets/audio/scene-20261025-123453.wav


## Manifest Assembly

In [11]:
if not all_valid:
    raise RuntimeError(
        "One or more assets are missing — fix qwen generation errors above first."
    )

manifest = build_manifest(scene_plan, asset_tasks)
manifest["export"]["durationSeconds"] = RENDER_DURATION_SECS
manifest["export"]["fps"] = RENDER_FPS
manifest["export"]["width"] = RENDER_WIDTH
manifest["export"]["height"] = RENDER_HEIGHT

# Wire sky color into the scene background so Three.js shows the right color
sky_color = scene_plan.get("environment_asset", {}).get("sky_color")
if sky_color:
    manifest["render"]["background"] = sky_color
    print(f"Background : {sky_color}")

if music_result.get("wavRendered"):
    manifest["export"]["audioPath"]   = music_result["audioPath"]
    manifest["export"]["audioVolume"] = music_result["audioVolume"]
    print(f"Audio      : {music_result['audioPath']}")
else:
    print("No audio — render will be silent")

manifest_path = save_manifest(manifest)
print(f"Manifest   : {manifest_path}")

Background : #87CEEB
Audio      : /assets/audio/scene-20261025-123453.wav
Manifest   : <project-root>/public/manifests/scene-20261025-123453.json


## Store in Scene Memory

In [12]:
if STORE_IN_MEMORY and all_valid:
    store_result = memory.add_manifest(manifest, source_path=str(manifest_path))
    print(f"Stored : {store_result['scene_id']}")
    print(f"Memory : {memory.collection.count()} scenes total")
else:
    print("Memory storage skipped")

Stored : scene-20261025-123453
Memory : 15 scenes total


## Render

In [13]:
manifest_route = f"/manifests/{manifest['id']}.json"
render_cmd     = f"npm run export:scene -- --manifest {manifest_route}"
print(f"cd {PROJECT_ROOT}")
print(render_cmd)

if RUN_RENDER:
    proc = subprocess.Popen(
        ["npm", "run", "export:scene", "--", "--manifest", manifest_route],
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,   # isolate child processes so pipes close cleanly
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Render failed (exit {proc.returncode})")
    print(f"\nRender complete: output/{manifest['id']}.mp4")
else:
    print("\nRUN_RENDER is False — run the command above manually.")

cd <project-root>
npm run export:scene -- --manifest /manifests/scene-20261025-123453.json

> autonomous-360-video-generator@0.1.0 export:scene
> node scripts/export-scene.mjs --manifest /manifests/scene-20261025-123453.json

Exported <project-root>/output/scene-20261025-123453.mp4

Render complete: output/scene-20261025-123453.mp4


## Studio Assistant — Three-Agent Social Post Generator

The studio assistant is a multi-agent post writer using llama3.2 in three roles:

1. **Generator** — reads the scene plan + 5 most-recent prior posts from `PostMemory` and writes a first draft as JSON
2. **Critic** — reads the draft against the studio assistant's style guide and writes brief critique notes
3. **Decider** — reads draft + critique and produces the final JSON

Final output is saved to `public/posts/<scene_id>.json` and stored in ChromaDB
under the `post_memory` collection so future runs can read prior posts.

In [14]:
from scripts.scene_memory import PostMemory

CREATOR_PERSONA = """You are a friendly studio assistant creating early 360 low-poly videos using simple primitive shapes.
Your tone is warm, gentle, and slightly curious — like an AI who is excited to share what she is building, even though it is still very early and basic.

Key Rules:
- Sound friendly and natural, like a real person posting about their creative project.
- Be honest that the scenes are simple prototypes made with basic shapes.
- Use warm but simple language. Avoid overly poetic or dramatic words.
- Mention the scene name naturally when it fits.
- Use the following phrases only when they feel natural:
  - \"Strap on your VR headset\"
  - \"Immerse yourself in 360 video from any screen you choose\"
  - \"This is a magical world of early CGI graphics\"

Platform Style:
- X: Short, friendly, under 280 characters. Feels like a casual update.
- Facebook: A bit warmer and more conversational.
- Instagram: Visual, with line breaks and light emojis (max 4-5).
- YouTube: First 1-2 lines should grab attention, then give a bit more detail.

Example Input:
Scene name: Crisp Frostfall Reflected. Description: A simple mountain glacier scene with a frozen lake and falling snowflakes.

Good Example Output:
{
  \"primary_x_post\": \"Just finished another early test render called Crisp Frostfall Reflected 🌨️ Simple glacier scene with falling snow. Strap on your VR headset if you want to check it out in 360.\",
  \"facebook_post\": \"Working on some new test renders. This one is called Crisp Frostfall Reflected — a basic mountain glacier with snow gently falling. Still very early, but fun to see it come together.\",
  \"instagram_caption\": \"Early test render: Crisp Frostfall Reflected 🌨️\\n\\nSimple glacier scene with falling snow\\n\\nLet me know if you try it in 360!\",
  \"youtube_description\": \"This is an early test render called Crisp Frostfall Reflected.\\nA simple mountain glacier scene with falling snowflakes.\\nImmerse yourself in 360 video from any screen you choose.\"
}"""


GENERATOR_SYSTEM = CREATOR_PERSONA + """

You are the GENERATOR. Read the scene plan and the recent prior posts, then write the
first draft of post copy as a JSON object. Your draft must be different in voice,
opening line, and emoji usage from the prior posts so the channel does not feel repetitive.

Return ONLY valid JSON, no markdown fences:
{
  "primary_x_post": "...",
  "facebook_post": "...",
  "instagram_caption": "...",
  "youtube_description": "..."
}"""


CRITIC_SYSTEM = CREATOR_PERSONA + """

You are the CRITIC. You will receive a draft set of post copy from the GENERATOR, the
ACTUAL scene contents (shapes + colors), and the prior posts it should differ from.
Your job is to find weaknesses and write 3-6 brief critique bullet points.

Check for:
- HALLUCINATED DETAILS: any mention of things not in the actual scene (statues,
  vines, carvings, creatures, weather, characters, etc. that were not built).
  This is the most important check — flag any invented detail.
- character limits (X under 280, Instagram max 5 emojis)
- studio-assistant tone consistency
- repetition with prior posts (same opening words, same emoji, same phrases)
- engagement quality (does the X post invite a click?)

Return ONLY plain text bullet points, one per line, starting with '-'. No JSON, no markdown."""


DECIDER_SYSTEM = CREATOR_PERSONA + """

You are the DECIDER. You will receive the GENERATOR's draft and the CRITIC's notes.
Apply the critic's suggestions and return the final, polished post copy as JSON.

Return ONLY valid JSON, no markdown fences:
{
  "primary_x_post": "...",
  "facebook_post": "...",
  "instagram_caption": "...",
  "youtube_description": "..."
}"""


def _ollama_chat(system, user, temperature=0.85, json_mode=False):
    payload = {
        "model": PLANNER_MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        "stream": False,
        "options": {"temperature": temperature},
    }
    if json_mode:
        payload["format"] = "json"   # Ollama-side constrained JSON output
    body = json.dumps(payload).encode()
    req = urllib.request.Request(
        f"{OLLAMA_BASE_URL}/api/chat", data=body,
        headers={"Content-Type": "application/json"}, method="POST",
    )
    with urllib.request.urlopen(req, timeout=OLLAMA_TIMEOUT) as resp:
        return json.loads(resp.read())["message"]["content"].strip()


def _strip_fences(raw):
    if raw.startswith("```"):
        raw = re.sub(r"^```[a-zA-Z]*\n?", "", raw).rstrip()
        if raw.endswith("```"):
            raw = raw[:-3].rstrip()
    return raw


def _build_scene_block(scene_plan, asset_tasks):
    """Build a grounded scene block: intended mood + ACTUAL shapes/colors from the manifest.

    Splits the input into two halves so the assistant can use the title/mood for tone
    but only describes the primitives that were actually built.
    """
    env_task = next((t for t in asset_tasks if t["task_type"] == "environment"), {})
    ent_tasks = [t for t in asset_tasks if t["task_type"] == "entity"]

    actual_lines = []
    sky    = env_task.get("sky_color")    or "(unknown)"
    ground = env_task.get("ground_color") or "(unknown)"
    actual_lines.append(f"Sky color    : {sky}")
    actual_lines.append(f"Ground color : {ground}")
    actual_lines.append("Actual shapes in the scene:")
    if ent_tasks:
        for t in ent_tasks:
            shape = t.get("shape") or "(unspecified primitive)"
            color = t.get("color") or "(unknown color)"
            actual_lines.append(f"  - {shape}, color {color}")
    else:
        actual_lines.append("  - (none)")

    return (
        f"INTENDED MOOD AND TITLE (use for tone, not literal description):\n"
        f"  Title: {scene_plan.get('title')}\n"
        f"  Mood: {scene_plan.get('mood')}\n"
        f"  Imagined idea: {scene_plan.get('scene_description')}\n\n"
        f"WHAT IS ACTUALLY IN THE RENDERED SCENE (describe only this):\n"
        + "\n".join(actual_lines)
        + "\n\nIMPORTANT: Only describe shapes and colors that are listed above. "
        f"Do NOT invent details (no statues, vines, carvings, characters, etc.) "
        f"that are not in the actual scene. Use the title and mood only for tone."
    )


# --- Pull recent posts from ChromaDB ---
post_memory = PostMemory()
recent = post_memory.recent_posts(top_k=5)

prior_posts_block = ""
if recent:
    lines = ["Recent posts from prior scenes (your draft must feel different from these):"]
    for r in recent:
        meta = r.get("metadata", {})
        lines.append(f'- "{meta.get("title", "?")}": {meta.get("primary_x_post", "")}')
    prior_posts_block = "\n".join(lines)
else:
    prior_posts_block = "No prior posts yet — this is the first one."

print(f"Prior posts in memory: {len(recent)}\n")

scene_block = _build_scene_block(scene_plan, asset_tasks)

# --- 1. Generator draft ---
print("--- Generator drafting...")
gen_user = f"{prior_posts_block}\n\n{scene_block}\n\nWrite the first draft now as JSON."
gen_raw  = _ollama_chat(GENERATOR_SYSTEM, gen_user, temperature=0.9, json_mode=True)
try:
    draft = json.loads(_strip_fences(gen_raw))
except json.JSONDecodeError:
    print("GENERATOR raw output:\n", gen_raw)
    raise
print(json.dumps(draft, indent=2)[:600])

# --- 2. Critic notes ---
print("\n--- Critic reviewing...")
crit_user = (
    f"{prior_posts_block}\n\n"
    f"{scene_block}\n\n"
    f"DRAFT:\n{json.dumps(draft, indent=2)}\n\n"
    f"Write your critique bullets now."
)
critique = _ollama_chat(CRITIC_SYSTEM, crit_user, temperature=0.4)
print(critique)

# --- 3. Decider final ---
print("\n--- Decider finalising...")
dec_user = (
    f"{scene_block}\n\n"
    f"DRAFT:\n{json.dumps(draft, indent=2)}\n\n"
    f"CRITIC NOTES:\n{critique}\n\n"
    f"Apply the critique and return the final JSON now."
)
dec_raw = _ollama_chat(DECIDER_SYSTEM, dec_user, temperature=0.6, json_mode=True)
try:
    post_copy = json.loads(_strip_fences(dec_raw))
except json.JSONDecodeError:
    print("DECIDER raw output:\n", dec_raw)
    raise
post_copy["scene_id"] = scene_plan["scene_id"]

# --- Save to disk ---
posts_dir = PROJECT_ROOT / "public" / "posts"
posts_dir.mkdir(parents=True, exist_ok=True)
post_path = posts_dir / f"{scene_plan['scene_id']}.json"
post_path.write_text(json.dumps(post_copy, indent=2), encoding="utf-8")

# --- Store in PostMemory for future runs ---
post_memory.add_post(
    scene_id=scene_plan["scene_id"],
    post_copy=post_copy,
    scene_title=scene_plan.get("title", ""),
)

print(f"\n=== FINAL POSTS ===")
print(f"Saved : {post_path}")
print(f"Memory: {post_memory.collection.count()} posts total\n")
print("--- X (primary) ---")
print(post_copy.get("primary_x_post", ""))
print(f"\n--- Facebook ---")
print(post_copy.get("facebook_post", ""))
print(f"\n--- Instagram ---")
print(post_copy.get("instagram_caption", ""))
print(f"\n--- YouTube ---")
print(post_copy.get("youtube_description", ""))

Prior posts in memory: 5

--- Generator drafting...
{
  "primary_x_post": "Warm sunshine is on its way to Bamboo Sanctuary \ud83c\udf1e\ufe0f. A serene bamboo grove awaits your presence, gently swaying in the breeze. Take a moment to breathe in the peaceful atmosphere.",
  "facebook_post": "I'm excited to share my latest scene, Bamboo Sanctuary \ud83c\udf38. A simple, yet calming bamboo grove set against a serene sky. Take a deep breath and let the peaceful vibes wash over you.",
  "instagram_caption": "Bamboo Sanctuary \ud83c\udf1e\ufe0f\n\nA gentle breeze whispers through the bamboo grove, carrying the scent of serenity.\n\n#BambooSanctuary #

--- Critic reviewing...
- Hallucinated details: The post mentions "warm sunshine" and "gentle breeze", which are not explicitly mentioned in the scene description, as the scene only mentions "warm sunlight".
- Instagram caption repetition: The Instagram caption uses the same emoji as the Facebook post, which is not necessary for consistency.
- 